# cNMF Evaluation Pipeline Tutorial

This notebook demonstrates how to evaluate gene programs discovered by cNMF.

**Prerequisites**: You should have already run cNMF inference (prepare, factorize, combine, consensus)
and saved results as `.h5mu` MuData files (see the inference tutorial).

## Evaluation Steps

| Step | What it does |
|------|-------------|
| **Data validation** | Check that MuData has the required fields for evaluation |
| **Categorical association** | Test whether programs differ across conditions (e.g. cell types, batches) |
| **Gene-set enrichment** | Test overlap of top program genes with known pathways (Reactome, GO) |
| **Trait enrichment** | Test overlap with GWAS-linked genes from Open Targets |
| **Perturbation association** | Test whether CRISPR perturbations shift program scores (requires guide data) |
| **Explained variance** | Measure how much expression variance each program captures |
| **Motif enrichment** | Test enrichment of TF motifs in enhancers/promoters linked to program genes (optional) |

## 0. Setup

In [ ]:
import os
import sys
import pandas as pd
import scanpy as sc
import mudata as mu

# Add the pipeline root to the path
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline')

from torch_cnmf import cNMF

from Evaluation.src import (
    compute_categorical_association,
    compute_geneset_enrichment,
    compute_trait_enrichment,
    compute_perturbation_association,
    compute_explained_variance,
    compute_motif_enrichment,
)

## 1. Configure Paths and Parameters

Edit the paths below to point to your own data.

In [ ]:
# ── cNMF output paths ──
out_dir = "/path/to/your/cnmf/output"          # directory passed as output_dir to cNMF
run_name = "your_run_name"                       # name passed to cNMF

# ── Resource files ──
# Normalized counts produced by cNMF prepare (inside cnmf_tmp/)
X_normalized_path = f"{out_dir}/{run_name}/cnmf_tmp/{run_name}.norm_counts.h5ad"

# Open Targets GWAS data for trait enrichment
gwas_data_path = "/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline/Evaluation/Resources/OpenTargets_L2G_Filtered.csv.gz"

# (Optional) Guide annotation file — set to None if not using perturbation data
guide_annotation_path = None  # e.g. "/path/to/guide_metadata.tsv"

# ── Perturbation data (only needed for perturbation association) ──
# Path to the MuData or AnnData that contains guide assignment info
mdata_guide_path = None  # e.g. "/path/to/perturb_seq_data.h5ad"

In [ ]:
# ── MuData keys ──
# These must match the keys used when building the MuData in the inference step
data_key = "rna"              # key for the RNA expression modality
prog_key = "cNMF"             # key for the cNMF program modality
categorical_key = "Condition" # obs column with sample/condition labels
organism = "human"            # "human" or "mouse"

# Guide-related keys (only needed for perturbation association)
guide_assignment_key = "guide_assignment"
guide_names_key = "guide_names"
guide_targets_key = "guide_targets"

# Gene names key — column in adata.var with gene symbols (e.g. "symbol")
# Set to None to use var_names / uns['var_names'] with automatic Ensembl ID conversion
gene_names_key = "symbol"

# ── K and threshold grid to evaluate ──
components = [5, 7, 10]              # values of K that were run in inference
sel_threshs = [0.4, 0.8, 2.0]        # density thresholds used in consensus

## 2. Load Shared Data

These objects are reused across all K / threshold combinations.

In [ ]:
# cNMF object — needed for explained variance calculation
cnmf_obj = cNMF(output_dir=out_dir, name=run_name)

# Normalized expression matrix
X_norm = sc.read_h5ad(X_normalized_path)
X = X_norm.X

In [ ]:
# (Optional) Load guide assignment data for perturbation association
ad_guide = None
if mdata_guide_path is not None:
    ad_guide = mu.read(mdata_guide_path)
    print(f"Loaded guide data: {ad_guide.shape}")

## 3. Helper: Attach Guide Metadata to MuData

If you have perturbation (CRISPR screen) data, guide assignments need to be
copied into the MuData object so the perturbation association test can find them.

In [ ]:
def assign_guide_metadata(mdata, ad_guide, gene_names_key='symbol'):
    """Copy guide assignment info from a reference AnnData into the MuData."""
    mdata["cNMF"].uns["guide_targets"] = ad_guide.uns["guide_targets"]
    mdata["cNMF"].obsm["guide_assignment"] = ad_guide.obsm["guide_assignment"]
    mdata["rna"].var_names = ad_guide.var[gene_names_key].astype(str)


## 4. Run Evaluation Loop

This loop iterates over all (K, threshold) combinations and runs each evaluation step.
Results are saved as TSV files in `<out_dir>/<run_name>/Eval/<K>_<threshold>/`.

You can comment out any section you don't need.

In [ ]:
for sel_thresh in sel_threshs:
    thresh_str = str(sel_thresh).replace(".", "_")

    for k in components:
        print(f"\n{'='*60}")
        print(f"Evaluating K={k}, density_threshold={sel_thresh}")
        print(f"{'='*60}")

        output_folder = f"{out_dir}/{run_name}/Eval/{k}_{thresh_str}"
        os.makedirs(output_folder, exist_ok=True)

        # ── Load MuData ──
        mdata_path = f"{out_dir}/{run_name}/adata/cNMF_{k}_{thresh_str}.h5mu"
        mdata = mu.read(mdata_path)

        # Attach guide metadata if available
        if ad_guide is not None:
            assign_guide_metadata(mdata, ad_guide, gene_names_key=gene_names_key)

        # ────────────────────────────────────────────────────────────
        # 4a. Categorical Association
        #     Tests whether program scores differ across conditions
        #     (Kruskal-Wallis + posthoc pairwise tests)
        # ────────────────────────────────────────────────────────────
        print("Running categorical association...")
        results_df, posthoc_df = compute_categorical_association(
            mdata,
            prog_key=prog_key,
            categorical_key=categorical_key,
            pseudobulk_key=None,
            test="dunn",
            n_jobs=-1,
            inplace=False,
        )
        results_df.to_csv(f"{output_folder}/{k}_categorical_association_results.txt", sep="\t", index=False)
        posthoc_df.to_csv(f"{output_folder}/{k}_categorical_association_posthoc.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 4b. Gene-Set Enrichment (Reactome)
        #     Fisher test of top program genes against Reactome pathways
        # ────────────────────────────────────────────────────────────
        print("Running Reactome gene-set enrichment...")
        reactome_res = compute_geneset_enrichment(
            mdata,
            prog_key=prog_key,
            data_key=data_key,
            organism=organism,
            library="Reactome_2022",
            method="fisher",
            database="enrichr",
            n_top=300,
            n_jobs=-1,
            inplace=False,
            use_loadings_gene=True,
            gene_names_key=gene_names_key,
        )
        reactome_res.to_csv(f"{output_folder}/{k}_geneset_enrichment_reactome.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 4c. Gene-Set Enrichment (GO Biological Process)
        #     Same as above but against GO terms
        # ────────────────────────────────────────────────────────────
        print("Running GO Biological Process enrichment...")
        go_res = compute_geneset_enrichment(
            mdata,
            prog_key=prog_key,
            data_key=data_key,
            organism=organism,
            library="GO_Biological_Process_2023",
            method="fisher",
            database="enrichr",
            n_top=300,
            n_jobs=-1,
            inplace=False,
            use_loadings_gene=True,
            gene_names_key=gene_names_key,
        )
        go_res.to_csv(f"{output_folder}/{k}_GO_term_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 4d. Trait Enrichment (GWAS)
        #     Fisher test of top program genes against Open Targets
        #     GWAS locus-to-gene mappings
        # ────────────────────────────────────────────────────────────
        print("Running GWAS trait enrichment...")
        trait_res = compute_trait_enrichment(
            mdata,
            gwas_data=gwas_data_path,
            prog_key=prog_key,
            data_key=data_key,
            library="OT_GWAS",
            n_jobs=-1,
            inplace=False,
            key_column="trait_efos",
            gene_column="gene_name",
            method="fisher",
            n_top=300,
            use_loadings_gene=False,
            gene_names_key=gene_names_key,
        )
        trait_res.to_csv(f"{output_folder}/{k}_trait_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 4e. Perturbation Association (requires guide data)
        #     Mann-Whitney U test: do perturbed cells have shifted
        #     program scores vs non-targeting controls?
        #     Runs separately per condition/sample.
        # ────────────────────────────────────────────────────────────
        if ad_guide is not None:
            print("Running perturbation association...")
            for samp in mdata["rna"].obs[categorical_key].unique():
                mdata_sub = mdata[mdata["rna"].obs[categorical_key] == samp]
                test_stats_df = compute_perturbation_association(
                    mdata_sub,
                    prog_key=prog_key,
                    collapse_targets=True,
                    pseudobulk=False,
                    reference_targets=["Non-targeting"],
                    FDR_method="StoreyQ",
                    n_jobs=-1,
                    inplace=False,
                )
                test_stats_df.to_csv(
                    f"{output_folder}/{k}_perturbation_association_{samp}.txt", sep="\t", index=False
                )
        else:
            print("Skipping perturbation association (no guide data provided).")

        # ────────────────────────────────────────────────────────────
        # 4f. Explained Variance
        #     How much of the normalized expression variance is
        #     captured by the consensus W and H matrices?
        # ────────────────────────────────────────────────────────────
        print("Computing explained variance...")
        compute_explained_variance(
            cnmf_obj,
            X,
            k,
            output_folder=output_folder,
            thre=str(sel_thresh),
            program_name=mdata[prog_key].var_names,
        )

        print(f"Done. Results saved to {output_folder}")

## Output Files

After running, each `<out_dir>/<run_name>/Eval/<K>_<threshold>/` folder contains:

| File | Description |
|------|-------------|
| `*_categorical_association_results.txt` | Kruskal-Wallis test results per program |
| `*_categorical_association_posthoc.txt` | Pairwise posthoc (Dunn) test results |
| `*_geneset_enrichment_reactome.txt` | Reactome pathway enrichment (Fisher) |
| `*_GO_term_enrichment.txt` | GO Biological Process enrichment (Fisher) |
| `*_trait_enrichment.txt` | GWAS trait enrichment from Open Targets |
| `*_perturbation_association_<sample>.txt` | Per-sample perturbation U-test results |
| `*_explained_variance.tsv` | Variance explained by each program |